Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV

Data Preprocessing

In [2]:
# Load dataset
df = pd.read_csv('../datasets/Lab 8 - Sheet1.csv')

# Drop index/ID column if present
if 'No' in df.columns:
    df = df.drop(columns=['No'])

# Separate input features and target variable
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']]
y = df['Play Tennis']

# Encode categorical features and target labels
feature_encoder = OrdinalEncoder()
X_encoded = feature_encoder.fit_transform(X)

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print("Features processed successfully.")
print("Feature Categories:")
for col, cats in zip(X.columns, feature_encoder.categories_):
    print(f"  - {col}: {list(cats)}")
print(f"Target Classes: {list(target_encoder.classes_)}")

Features processed successfully.
Feature Categories:
  - Outlook: ['Overcast', 'Rain', 'Sunny']
  - Temperature: ['Cool', 'Hot', 'Mild']
  - Humidity: ['High', 'Normal']
  - Wind: ['Strong', 'Weak']
Target Classes: ['No', 'Yes']


Data Partitioning

In [3]:
# 80:20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Total Samples : {len(df)}")
print(f"Training Set  : {X_train.shape[0]} samples")
print(f"Testing Set   : {X_test.shape[0]} samples")

Total Samples : 50
Training Set  : 40 samples
Testing Set   : 10 samples


Naive Bayes Model Training and Evaluation

In [4]:
# Train Categorical Naive Bayes model
cnb = CategoricalNB()
cnb.fit(X_train, y_train)

# Predict test set
y_pred_nb = cnb.predict(X_test)

# Calculate Accuracy & Evaluation Metrics
acc_nb = accuracy_score(y_test, y_pred_nb)
cm_nb = confusion_matrix(y_test, y_pred_nb)
report_nb = classification_report(y_test, y_pred_nb, target_names=target_encoder.classes_)

print(f"Overall Model Accuracy: {acc_nb * 100:.2f}%\n")
print("Confusion Matrix:")
print(cm_nb)
print("\nClassification Report:")
print(report_nb)

Overall Model Accuracy: 90.00%

Confusion Matrix:
[[2 1]
 [0 7]]

Classification Report:
              precision    recall  f1-score   support

          No       1.00      0.67      0.80         3
         Yes       0.88      1.00      0.93         7

    accuracy                           0.90        10
   macro avg       0.94      0.83      0.87        10
weighted avg       0.91      0.90      0.89        10



Single-Sample Inference

In [5]:
# Define custom query: Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong
single_sample = pd.DataFrame([{
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}])

# Transform sample using fitted encoder
single_sample_encoded = feature_encoder.transform(single_sample)

# Predict class label and probability scores
sample_pred_code = cnb.predict(single_sample_encoded)[0]
sample_pred_label = target_encoder.inverse_transform([sample_pred_code])[0]
sample_proba_nb = cnb.predict_proba(single_sample_encoded)[0]

print(f"Custom Query Input: {single_sample.to_dict(orient='records')[0]}")
print(f"Predicted Class Label: {sample_pred_label}")
print("Class Probabilities:")
for label, prob in zip(target_encoder.classes_, sample_proba_nb):
    print(f"  - {label}: {prob:.4f} ({prob*100:.2f}%)")

Custom Query Input: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class Label: No
Class Probabilities:
  - No: 0.7894 (78.94%)
  - Yes: 0.2106 (21.06%)


Model Comparison

In [6]:
# Define models to compare
models = {
    'Categorical Naive Bayes': CategoricalNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM': CalibratedClassifierCV(estimator=SVC(random_state=42), ensemble=False)
}

comparison_results = []

for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)
    
    # Predict test accuracy
    test_acc = accuracy_score(y_test, model.predict(X_test))
    
    # Predict on custom single sample
    sample_pred = model.predict(single_sample_encoded)[0]
    sample_label = target_encoder.inverse_transform([sample_pred])[0]
    sample_proba = model.predict_proba(single_sample_encoded)[0]
    
    comparison_results.append({
        'Model': name,
        'Test Accuracy': f"{test_acc * 100:.2f}%",
        'Single Sample Prediction': sample_label,
        'P(No)': f"{sample_proba[0]:.4f}",
        'P(Yes)': f"{sample_proba[1]:.4f}"
    })

# Format into DataFrame table
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

                  Model Test Accuracy Single Sample Prediction  P(No) P(Yes)
Categorical Naive Bayes        90.00%                       No 0.7894 0.2106
          Decision Tree       100.00%                       No 1.0000 0.0000
    Logistic Regression        90.00%                       No 0.8613 0.1387
                    SVM       100.00%                       No 0.9415 0.0585
